# Marketing Mix Registry Walkthrough

This notebook walks through registering a marketing mix model configuration with the Model Zoo registry API and executing a run using synthetic data. Every step mirrors the command-line helper but keeps the process transparent and reproducible.

## Prerequisites

- The Docker Compose stack (`docker compose up --build`) should be running. This provides the FastAPI registry at `http://localhost:8000` and PostgreSQL storage.
- The repository root is mounted in this notebook session so we can load the bundled configuration (`configs/marketing_mix.json`).
- Python package `requests` is available. The following cell installs it if missing.

In [ ]:
#!pip install requests --quiet

## 1. Configure API Target

Set the base URL for the API and paths to the configuration file. Adjust these if your registry runs elsewhere.

In [2]:
import json
from pathlib import Path
import requests

BASE_URL = "http://localhost:8000"

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd().resolve()

search_roots = [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents)
cwd = Path.cwd().resolve()
search_roots.extend([cwd] + list(cwd.parents))

CONFIG_PATH = None
for root in search_roots:
    candidate = root / "configs" / "marketing_mix.json"
    if candidate.exists():
        CONFIG_PATH = candidate.resolve()
        break

if CONFIG_PATH is None:
    checked = ", ".join(str(r) for r in dict.fromkeys(search_roots))
    raise FileNotFoundError(f"Could not locate configs/marketing_mix.json (checked: {checked})")

ENV = "dev"  # environment label stored with the configuration

print(f"API base URL: {BASE_URL}")
print(f"Using configuration: {CONFIG_PATH}")
print(f"Environment tag: {ENV}")

FileNotFoundError: Expected config at configs/marketing_mix.json

## 2. Register the Marketing Mix Configuration

The configuration contains:
- Model metadata and binding (`model` section)
- Hyperparameter schema and engine values (`hyperparameters`)
- Metric definitions and bindings (`metrics`)
- Variable definitions and optional groupings (`variables`)
- Default evaluation plan (`evaluation`)
- Execution context hints (`context`)

We POST this JSON to `/registry/configs`. The API stores it in Postgres, registers the model/metrics, and keeps the bindings in memory.

### Inspect Coefficient Priors

Coefficients are configured separately from hyperparameters. The `intercept`
entry controls the global baseline, while each entry in `variables` maps a
feature name (matching the transformed dataset) to its prior distribution.
Editing this section lets modelers add or remove variables without touching the
runner code.

In [ ]:
config_payload.get("coefficient_priors", {})

### Coefficient Tables

The base configuration pulls variable priors from `configs/marketing_mix_coefficients.csv`.
You can edit that CSV (or point to an alternate table) to add/remove variables without
changing the JSON payload.

In [ ]:
config_payload = json.loads(CONFIG_PATH.read_text())
response = requests.post(
    f"{BASE_URL.rstrip('/')}/registry/configs",
    json={"env": ENV, "config": config_payload},
    timeout=30,
)
response.raise_for_status()
registered = response.json()
registered

## 3. Inspect the Stored Specification

Fetch the persisted model spec and associated configurations using `/registry/models/{model_id}`. This confirms the registry recorded hyperparameters, tags, and metric bindings.

In [ ]:
model_id = registered.get("model_id")
if not model_id:
    raise ValueError("Registry response missing model_id")

spec_response = requests.get(f"{BASE_URL.rstrip('/')}/registry/models/{model_id}", timeout=30)
spec_response.raise_for_status()
model_spec = spec_response.json()
model_spec

## 4. Generate Synthetic Marketing Mix Data

Recreate the helper from `bhm_pymc/marketing_mix_pymc_example.py`. The generator produces weekly channel spend for TV/Radio/Social and corresponding sales values. The data sources must match the `variables.definitions` in the configuration (e.g., `tv_spend`, `marketing_rows`, `sales`).

In [ ]:
from typing import List, Dict, Tuple
import random


def make_synthetic_marketing_mix_weeks(n_weeks: int = 52, seed: int = 42) -> Tuple[List[Dict[str, float]], List[float]]:
    random.seed(seed)
    tv = [random.uniform(5.0, 30.0) for _ in range(n_weeks)]
    radio = [random.uniform(1.0, 10.0) for _ in range(n_weeks)]
    social = [random.uniform(0.5, 8.0) for _ in range(n_weeks)]

    alpha = 100.0
    b_tv, b_radio, b_social = 2.0, 1.0, 0.5
    noise_sigma = 5.0

    sales = [alpha + b_tv * tv[i] + b_radio * radio[i] + b_social * social[i] + random.gauss(0, noise_sigma) for i in range(n_weeks)]
    rows = [{"tv": tv[i], "radio": radio[i], "social": social[i]} for i in range(n_weeks)]
    return rows, sales

rows, sales = make_synthetic_marketing_mix_weeks()
len(rows), rows[0], sales[0]

In [ ]:
import time

def poll_run(base_url: str, run_id: str, timeout: float = 600.0, interval: float = 0.5):
    """Poll the /runs/{id} endpoint until the run reaches a terminal state."""
    deadline = time.time() + timeout
    while True:
        resp = requests.get(f"{base_url.rstrip('/')}/runs/{run_id}", timeout=30)
        resp.raise_for_status()
        payload = resp.json()
        status = payload.get("status")
        if status in {"succeeded", "failed", "cancelled"}:
            return payload
        if time.time() >= deadline:
            raise TimeoutError(f"Run {run_id} did not finish within {timeout} seconds")
        time.sleep(interval)


## 5. Submit a Run via `/runs`

Construct the payload expected by the new run endpoint:
- `model_id` and `env` select the stored configuration.
- `data_sources` provides the raw values referenced in the config (marketing rows, channel spends, and sales targets).
- `evaluation` overrides metric IDs if desired (here we re-use the config defaults).

The endpoint executes the model in-process using the registered binding, computes the metrics, and returns the summary.

In [ ]:
from itertools import repeat

# Build data sources to match the variable definitions (see configs/marketing_mix.json)
tv_spend = [row["tv"] for row in rows]
radio_spend = [row["radio"] for row in rows]
social_spend = [row["social"] for row in rows]

run_payload = {
    "model_id": model_id,
    "env": ENV,
    "data_sources": {
        "marketing_rows": rows,
        "sales": sales,
        "tv_spend": tv_spend,
        "radio_spend": radio_spend,
        "social_spend": social_spend,
    },
    "evaluation": {
        "metric_ids": ["r2_v1", "mse_v1"],
        "y_true_key": "y",
    }
}

submission = requests.post(f"{BASE_URL.rstrip('/')}/runs", json=run_payload, timeout=30)
submission.raise_for_status()
run_ticket = submission.json()
print(f"Submitted run {run_ticket['run_id']} (status={run_ticket['status']})")

status_payload = poll_run(BASE_URL, run_ticket['run_id'])
run_result = status_payload.get("result", {})
run_result


In [ ]:
run_result["diagnostics"]["coefficients_model"]

In [ ]:
run_result["diagnostics"]["param_values"]

## 6. Review Evaluation Metrics and Summary

The response includes:
- `status`: `succeeded`, `failed`, or `cancelled`
- `diagnostics`: coefficients and intercept emitted by the runner
- `eval_train` / `eval_valid`: metric scores on the splits
- `summary_text`: R-style summary (also available in `summary` as a dict)

Use this cell to display key pieces in a friendly format.

In [ ]:
from pprint import pprint

status = run_result.get("status") or status_payload.get("status")
print(f"Run status: {status}")
print("
Train metrics:")
pprint(run_result.get("eval_train", {}))
print("
Validation metrics:")
pprint(run_result.get("eval_valid", {}))

print("
Summary text:")
print(run_result.get("summary_text"))

### Optional: Adjust Configuration (Temporary)

You can experiment with coefficient or hyperparameter tweaks by copying the
configuration into memory, editing it, and submitting a run without touching the
base JSON file. The following cells demonstrate removing a variable and changing
the sigma scale, then restoring the original payload.

In [ ]:
from copy import deepcopy

config_custom = deepcopy(config_payload)
priors_section = config_custom.setdefault("coefficient_priors", {})
variables_priors = priors_section.setdefault("variables", [])
priors_section["intercept"] = {
    "distribution": "normal",
    "params": {"mu": 95.0, "sigma": 15.0},
    "constraints": {"sign": "positive"}
}
# remove radio and digital if present, then append new priors
priors_section["variables"] = [
    var for var in variables_priors if var.get("name") not in {"radio", "digital"}
]
priors_section["variables"].extend([
    {
        "name": "tv",
        "distribution": "normal",
        "params": {"mu": 2.2, "sigma": 0.9},
        "constraints": {"sign": "positive"}
    },
    {
        "name": "social",
        "distribution": "gamma",
        "params": {"shape": 2.5, "rate": 1.0},
        "constraints": {"lower": 0.0}
    },
    {
        "name": "digital",
        "distribution": "normal",
        "params": {"mu": 0.6, "sigma": 0.3},
        "constraints": {"lower": 0.0}
    }
])

# add variable definition for digital spend sourced from computed column
variables_section = config_custom.setdefault("variables", {}).setdefault("definitions", [])
if not any(defn.get("name") == "digital" for defn in variables_section):
    variables_section.append({
        "name": "digital",
        "source": "digital_spend",
        "include": True,
        "transforms": [{"id": "identity"}]
    })

print("Modified coefficient priors (inline view):")
priors_section


In [ ]:
# Register the temporary configuration under a separate environment
experiment_env = "dev-experiment"
experiment_payload = {
    "env": experiment_env,
    "model_id": model_id,
    "config": config_custom,
}
_ = requests.post(f"{BASE_URL.rstrip('/')}/registry/configs", json=experiment_payload, timeout=30)

# Extend data sources with synthetic digital spend
digital_spend = [row["tv"] * 0.3 + row["social"] * 0.4 for row in rows]
custom_data_sources = dict(run_payload["data_sources"])
custom_data_sources["digital_spend"] = digital_spend
custom_data_sources["marketing_rows"] = [
    {**row, "digital": digital_spend[i]}
    for i, row in enumerate(custom_data_sources["marketing_rows"])
]

custom_submission = requests.post(
    f"{BASE_URL.rstrip('/')}/runs",
    json={
        "model_id": model_id,
        "env": experiment_env,
        "data_sources": custom_data_sources,
        "evaluation": run_payload["evaluation"],
    },
    timeout=30,
)
custom_submission.raise_for_status()
custom_ticket = custom_submission.json()
print(f"Submitted experiment run {custom_ticket['run_id']} (status={custom_ticket['status']})")
custom_status = poll_run(BASE_URL, custom_ticket['run_id'])
custom_result = custom_status.get("result", {})
custom_result.get("diagnostics", {}).get("coefficients_model", custom_result.get("diagnostics", {}))


## Next Steps

- Experiment with alternate configurations (adjust coefficients, transforms, or metrics) and re-run the notebook.
- Swap the synthetic generator for real weekly marketing data by populating the `data_sources` dict accordingly.
- Extend the API client with authentication or asynchronous run submission if you need production-grade orchestration.